# GridWorld 3×4 — Policy Iteration (Dynamic Programming)

**Example**: grildworld 3*4

**Problem**: A robot need to go from start to target

**Reward**: +1 for target goal, -1 for target hole, -0.04 for each step

**transition probabilities**: 
- 0.8 for the intended direction
- 0.1 for slipping to on side, and
- 0.1 for slipping to the other side

**Directions**:
- 0 to the left
- 1 downwards
- 2 to the right
- 3 upwards


<table style="border-collapse: collapse; table-layout: fixed; width: 320px;">
    <tr>
        <td style="border: 2px solid black; width: 80px; height: 80px; vertical-align: top; text-align: left;">S0</td>
        <td style="border: 2px solid black; width: 80px; height: 80px; vertical-align: top; text-align: left;">S1</td>
        <td style="border: 2px solid black; width: 80px; height: 80px; vertical-align: top; text-align: left;">S2</td>
        <td style="border: 2px solid black; width: 80px; height: 80px; vertical-align: top; text-align: left;">
            S3
            <div style="text-align: center; font-size: 20px; font-weight: bold;">+1</div>
        </td>
    </tr>
    <tr>
        <td style="border: 2px solid black; width: 80px; height: 80px; vertical-align: top; text-align: left;">S4</td>
        <td style="border: 2px solid black; width: 80px; height: 80px; vertical-align: top; text-align: left; background-color: lightgray;">S5</td>
        <td style="border: 2px solid black; width: 80px; height: 80px; vertical-align: top; text-align: left;">S6</td>
        <td style="border: 2px solid black; width: 80px; height: 80px; vertical-align: top; text-align: left;">
            S7
            <div style="text-align: center; font-size: 20px; font-weight: bold;">-1</div>
        </td>
    </tr>
    <tr>
        <td style="border: 2px solid black; width: 80px; height: 80px; vertical-align: top; text-align: left;">S8</td>
        <td style="border: 2px solid black; width: 80px; height: 80px; vertical-align: top; text-align: left;">S9</td>
        <td style="border: 2px solid black; width: 80px; height: 80px; vertical-align: top; text-align: left;">S10</td>
        <td style="border: 2px solid black; width: 80px; height: 80px; vertical-align: top; text-align: left;">S11</td>
    </tr>
</table>

This notebook implements **Policy Iteration** in a 3×4 GridWorld environment.

## Import Libraries

We use:
- `gymnasium` to create the environment,
- `numpy` for numerical comparisons,
- `random` to break ties randomly when multiple actions have the same maximum value.

If Gymnasium is not installed, run this command in a separate cell:

```python
%pip install gymnasium numpy
```

In [1]:
import gymnasium as gym
import numpy as np
import random


## Create the Environment and Define the Transition Dynamics

Each transition is represented by a tuple:

`(probability, next_state, reward, terminated)`

For example, in state `0`, when taking action `Right`, the agent moves to state `1` with probability `0.8`. With probability `0.1`, it may remain in the current state or slip to state `4`.

In this implementation, we directly update the Gymnasium transition table `P` so that it matches the desired GridWorld dynamics.

In [2]:
class GridWorld3x4():
    
    def __init__(self):
        ##
        self.env = gym.make(
            "FrozenLake-v1",
            desc=["SFFG",
                  "FFFH",
                  "FFFF"],
            is_slippery=True,
        )
        
        
        # state 0 
        self.env.unwrapped.P[0][0] = [
            (0.8, 0, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 0, -0.04, False),
            (0.1, 4, -0.04, False)
        ]
        
        self.env.unwrapped.P[0][1] = [
            (0.8, 4, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 0, -0.04, False),
            (0.1, 1, -0.04, False)
        ]
        
        self.env.unwrapped.P[0][2] = [
            (0.8, 1, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 0, -0.04, False),
            (0.1, 4, -0.04, False)
        ]
        
        self.env.unwrapped.P[0][3] = [
            (0.8, 0, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 0, -0.04, False),
            (0.1, 1, -0.04, False)
        ]
        
        # state 1 
        self.env.unwrapped.P[1][0] = [
            (0.8, 0, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 1, -0.04, False),
            (0.1, 1, -0.04, False)
        ]
        self.env.unwrapped.P[1][1] = [
            (0.8, 1, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 0, -0.04, False),
            (0.1, 2, -0.04, False)
        ]
        self.env.unwrapped.P[1][2] = [
            (0.8, 2, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 1, -0.04, False),
            (0.1, 1, -0.04, False)
        ]
        self.env.unwrapped.P[1][3] = [
            (0.8, 1, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 0, -0.04, False),
            (0.1, 2, -0.04, False)
        ]
        
        # state 2 
        self.env.unwrapped.P[2][0] = [
            (0.8, 1, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 2, -0.04, False),
            (0.1, 6, -0.04, False)
        ]
        self.env.unwrapped.P[2][1] = [
            (0.8, 6, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 1, -0.04, False),
            (0.1, 3, 1-0.04, True)
        ]
        self.env.unwrapped.P[2][2] = [
            (0.8, 3, 1-0.04, True), # (prob, next_state, reward, terminated)
            (0.1, 2, -0.04, False),
            (0.1, 6, -0.04, False)
        ]
        self.env.unwrapped.P[2][3] = [
            (0.8, 2, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 3, 1-0.04, True),
            (0.1, 1, -0.04, False)
        ]
        
        # state 3 
         # Terminal
         
        # state 4 
        self.env.unwrapped.P[4][0] = [
            (0.8, 4, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 0, -0.04, False),
            (0.1, 8, -0.04, False)
        ]
        self.env.unwrapped.P[4][1] = [
            (0.8, 8, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 4, -0.04, False),
            (0.1, 4, -0.04, False)
        ]
        self.env.unwrapped.P[4][2] = [
            (0.8, 4, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 0, -0.04, False),
            (0.1, 8, -0.04, False)
        ]
        self.env.unwrapped.P[4][3] = [
            (0.8, 0, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 4, -0.04, False),
            (0.1, 4, -0.04, False)
        ]
        
        # state 5 
            # Wall
            
        # state 6 
        self.env.unwrapped.P[6][0] = [
            (0.8, 6, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 2, -0.04, False),
            (0.1, 10, -0.04, False)
        ]
        self.env.unwrapped.P[6][1] = [
            (0.8, 10, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 6, -0.04, False),
            (0.1, 7, -1-0.04, True)
        ]
        self.env.unwrapped.P[6][2] = [
            (0.8, 7, -1-0.04, True), # (prob, next_state, reward, terminated)
            (0.1, 2, -0.04, False),
            (0.1, 10, -0.04, False)
        ]
        self.env.unwrapped.P[6][3] = [
            (0.8, 2, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 6, -0.04, False),
            (0.1, 7, -1-0.04, True)
        ]
        
        # state 7 
            # Terminal
            
        # state 8 
        self.env.unwrapped.P[8][0] = [
            (0.8, 8, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 4, -0.04, False),
            (0.1, 8, -0.04, False)
        ]
        self.env.unwrapped.P[8][1] = [
            (0.8, 8, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 8, -0.04, False),
            (0.1, 9, -0.04, False)
        ]
        self.env.unwrapped.P[8][2] = [
            (0.8, 9, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 4, -0.04, False),
            (0.1, 8, -0.04, False)
        ]
        self.env.unwrapped.P[8][3] = [
            (0.8, 4, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 9, -0.04, False),
            (0.1, 8, -0.04, False)
        ]
        
        # state 9 
        self.env.unwrapped.P[9][0] = [
            (0.8, 8, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 9, -0.04, False),
            (0.1, 9, -0.04, False)
        ]
        self.env.unwrapped.P[9][1] = [
            (0.8, 9, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 8, -0.04, False),
            (0.1, 10, -0.04, False)
        ]
        self.env.unwrapped.P[9][2] = [
            (0.8, 10, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 9, -0.04, False),
            (0.1, 9, -0.04, False)
        ]
        self.env.unwrapped.P[9][3] = [
            (0.8, 9, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 8, -0.04, False),
            (0.1, 10, -0.04, False)
        ]
        
        # state 10 
        self.env.unwrapped.P[10][0] = [
            (0.8, 9, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 6, -0.04, False),
            (0.1, 10, -0.04, False)
        ]
        self.env.unwrapped.P[10][1] = [
            (0.8, 10, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 9, -0.04, False),
            (0.1, 11, -0.04, False)
        ]
        self.env.unwrapped.P[10][2] = [
            (0.8, 11, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 6, -0.04, False),
            (0.1, 10, -0.04, False)
        ]
        self.env.unwrapped.P[10][3] = [
            (0.8, 6, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 11, -0.04, False),
            (0.1, 9, -0.04, False)
        ]
        
        # state 11 
        self.env.unwrapped.P[11][0] = [
            (0.8, 10, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 7, -1-0.04, True),
            (0.1, 11, -0.04, False)
        ]
        self.env.unwrapped.P[11][1] = [
            (0.8, 11, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 10, -0.04, False),
            (0.1, 11, -0.04, False)
        ]
        self.env.unwrapped.P[11][2] = [
            (0.8, 11, -0.04, False), # (prob, next_state, reward, terminated)
            (0.1, 7, -1-0.04, True),
            (0.1, 11, -0.04, False)
        ]
        self.env.unwrapped.P[11][3] = [
            (0.8, 7, -1-0.04, True), # (prob, next_state, reward, terminated)
            (0.1, 10, -0.04, False),
            (0.1, 11, -0.04, False)
        ]
        
        
        
    def step(self, state_initial, action): 
        ### state_inital
        self.env.reset()
        
        base = self.env.unwrapped
        base.s = state_initial 
        
        state, reward, terminated, truncated, info = self.env.step(action)

        return state, reward, terminated, truncated, info

    def reset(self,seed):
        self.env.reset(seed=seed)  

## Initialize the Policy and State-Value Function

`policy_pi[s]` stores the action selected by the current policy in state `s`.

`V[s]` estimates the expected return when starting from state `s` and following the current policy. Initially, all state values are set to zero.

Action indices follow the FrozenLake convention:
- `0`: Left
- `1`: Down
- `2`: Right
- `3`: Up

In [3]:
policy_pi = {
    (0): 2,
    (1): 2,
    (2): 2,
    (4): 3,
    (6): 2,
    (8): 3,
    (9): 2,
    (10): 3,
    (11): 3, 
} 

v = {
    (0): 0,
    (1): 0,
    (2): 0,
    (3): 0,
    (4): 0,
    (5): 0,
    (6): 0,
    (7): 0,
    (8): 0,
    (9): 0,
    (10): 0,
    (11): 0, 
} 

theta = 0.0001


### Convergence Parameters

`theta` determines the convergence threshold.

`delta` tracks the maximum change in the value function
during each iteration.

The algorithm stops when:

$$
\Delta < \theta
$$

In [7]:
########################## Parameters #####################
theta=0.001
delta = float("inf")

# enviroment
EnviGridWorld3x4 = GridWorld3x4()

P=EnviGridWorld3x4.env.unwrapped.P    


### Environment Transition Model

`P` contains the transition dynamics of the environment.

For each state-action pair, it stores the possible next states,
transition probabilities, rewards, and termination status.

The transition model is used by Dynamic Programming to evaluate
the value function without sampling episodes.

## Argmax with Random Tie-Breaking

The `argmax` function returns the index of the largest value. If multiple actions share the maximum value, one of them is selected randomly.

This avoids consistently favoring the same action when there is a tie.

In [15]:
def argmax(q,s):
    
    # When all Qs are equal, choose one at random from the 
    # maximum actions (which are all of them initially).    
    max_q_hat_s = max(q)
    
    best_actions=[]
    for i, v in enumerate(q):
        if np.isclose(v, max_q_hat_s):
            best_actions.append(i)
    
    
    best_action = random.choice(best_actions)
    
    return best_action


## Policy Iteration Algorithm

Policy Iteration repeatedly performs two steps:

### A. Policy Evaluation

While keeping the policy fixed, update the state-value function using the Bellman expectation equation:
$$
V(s) \leftarrow \sum_{s',r} p(s',r\mid s,\pi(s))
\left[r + V(s')\right]
$$
In this environment, the discount factor is `gamma = 1`, so it is omitted from the equation.

### B. Policy Improvement

For each state, calculate the expected value of every action:

$$
q(s,a) = \sum_{s',r} p(s',r\mid s,a)
\left[r + V(s')\right]
$$

Then select an action with the highest value. If no action changes, the policy is stable and the algorithm terminates.

**Important implementation detail:** `delta` must be reset to infinity at the beginning of every Policy Evaluation phase. Otherwise, after evaluating the first policy, a later evaluation phase might be skipped.

In [16]:
#################### Policy Iteration ####################

policy_stable=False
while policy_stable== False:
    
    # Policy Evaluation
    while delta >  theta:  # k: Gauss-seidel method
        
        delta=0
        for s in [0,1,2,4,6,8,9,10,11]:
            
            v_old= v[s]
    
            v_new=0
            a = policy_pi[s]
            
            for prob, s_prime, reward, done in P[s][a]:
                v_new= v_new+ prob*(reward+v[(s_prime)])
                
            v[(s)]=v_new
            
            # Maximum change in V(s) across all states
            delta = max(delta, abs(v_old - v_new))
       
            
    # Policy Improvement
    policy_stable = True
    for s in [0,1,2,4,6,8,9,10,11]:
    
        
        old_action = policy_pi[s]
        
        q=[]
        for a in [0,1,2,3]:
            
            q_s_a=0   
            for prob, s_prime, reward, done in P[s][a]:
                q_s_a= q_s_a+ prob*(reward+v[s_prime])
                   
            q.append(q_s_a)
        
        policy_pi[s]= argmax(q,s)
    
    
        if old_action != policy_pi[s]:
            policy_stable = False
            

## Display the Final Policy

The policy is displayed using directional symbols:
- `←` Left
- `↓` Down
- `→` Right
- `↑` Up

The resulting grid represents the stable policy found by Policy Iteration.

In [18]:
for s in [0,1,2,4,6,8,9,10,11]:
    
    if policy_pi[s]==0:
        policy_pi[s]="←"

    if policy_pi[s]==1:
        policy_pi[s]="↓"

    if policy_pi[s]==2:
        policy_pi[s]="→"

    if policy_pi[s]==3:
        policy_pi[s]="↑"

print("policy_pi_optimal=",policy_pi)



policy_pi_optimal= {0: '→', 1: '→', 2: '→', 4: '↑', 6: '↑', 8: '↑', 9: '←', 10: '↑', 11: '←'}


## Notes and Implementation Details

1. This notebook implements **Policy Iteration**, not Value Iteration.
2. Policy Evaluation uses in-place updates. A newly updated value can therefore be used when calculating later states in the same sweep. This is a Gauss–Seidel-style update.
3. `theta` is the stopping threshold used during Policy Evaluation.
4. The `terminated` flag is stored in the environment's transition table. The value of terminal states is zero, so the update naturally includes the terminal rewards (`+0.96` for the goal and `-1.04` for the hole).
5. To rerun the experiment from scratch, execute the notebook cells from top to bottom.
